# Materials for List 2: Tokenization & Embeddings

**Large Language Models** · Wrocław University of Science and Technology · Winter Term 2026/2027

Background for the tasks of List 2. Section numbers match the task numbers of the list. You can read it as a PDF, or run the notebook and experiment with the examples. The examples deliberately use different data from the tasks.

Lab 3 follows Sections 2.1–2.5 of Raschka's *Build a Large Language Model (From Scratch)* and Lecture 3; byte pair encoding for text is from Sennrich et al. (2016). Lab 4 follows Sections 2.6–2.8 of the book. Its background is written out in full here, because the Lecture 4 materials are not published yet: sinusoidal encodings are from Vaswani et al. (2017), skip-gram with negative sampling from Mikolov et al. (2013).

In [1]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn.functional as F
from torch import nn

print("PyTorch", torch.__version__, "| tiktoken", tiktoken.__version__)

PyTorch 2.14.1+cpu | tiktoken 0.14.0


## 0.1 Reading

- **Before Lab 3:** BLLM [Chapter 2](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02), Sections 2.1–2.5; [Lecture 3](../../lectures/03-tokenization/README.md), Parts 1–4 and 6
- **Before Lab 4:** BLLM Sections 2.6–2.8; Sections 4.1–4.4 below
- **Optional:** Sennrich et al. (2016), [*Neural Machine Translation of Rare Words with Subword Units*](https://arxiv.org/abs/1508.07909); Karpathy, ["Let's build the GPT Tokenizer"](https://www.youtube.com/watch?v=zduSFxRajkE) (video; Tasks 3.3–3.4 follow the same idea)

## 0.2 Setup

Python 3.10 or newer with `torch`, `tiktoken` and `matplotlib` (`pip install torch tiktoken matplotlib jupyterlab`). A GPU is not needed; [Google Colab](https://colab.research.google.com/) also works (if `tiktoken` is missing there, run `pip install tiktoken`). Everything in this list runs in a minute or two on a laptop CPU.

**One-time download.** [`tiktoken`](https://github.com/openai/tiktoken) is OpenAI's tokenizer library. The first call of `tiktoken.get_encoding("gpt2")` downloads GPT-2's vocabulary and merge files (about 1.5 MB) and caches them; after that it works offline. If you will be offline at the lab, run that line once beforehand.

**The text.** `the-verdict.txt` from List 1 is also in this folder (download: https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt). In scripts, put the two reusable definitions, `create_dataloader` and `GPTEmbedding`, in one module (say `list02.py`) and import them from the other scripts.

---

# Lab 3: Tokenizers from Scratch

## 3.1 A word tokenizer

### Tokens, vocabulary and IDs

A language model works with integers, not with text. A **tokenizer** converts between the two in two steps (Lecture 3, Part 1):

1. **split** the text into pieces, the **tokens**;
2. **look up** each token in the **vocabulary**, a fixed table that maps every known token to an integer **token ID** $0, \dots, V - 1$.

`encode` goes from text to IDs, `decode` from IDs back to text. In List 1 the tokens were single characters; here they are words and punctuation marks, and in Section 3.3 pieces of words.

### Boundary rules

Where one token ends and the next begins is a choice. Splitting at whitespace (`str.split()`) leaves punctuation attached: `Hello, Warsaw!` gives `Hello,` and `Warsaw!`, so `Warsaw!` and `Warsaw` would be different vocabulary entries. A **regular expression** gives finer control. `re.split(pattern, s)` cuts `s` at every match of `pattern`; if the pattern is wrapped in a capturing group `( … )`, the matches themselves are kept in the result. BLLM uses

```python
r'([,.:;?_!"()\']|--|\s)'
```

which matches one punctuation character, a double hyphen, or one whitespace character (`\s`: space, tab, newline). The result contains empty strings and whitespace pieces between neighbouring matches; keeping only the pieces with `t.strip()` non-empty leaves words and punctuation.

In [2]:
s = "Hello, Warsaw!  It's 9:30--isn't it?"
print(s.split())
pieces = re.split(r'([,.:;?_!"()\']|--|\s)', s)
print(pieces)
print([t for t in pieces if t.strip()])

['Hello,', 'Warsaw!', "It's", "9:30--isn't", 'it?']
['Hello', ',', '', ' ', 'Warsaw', '!', '', ' ', '', ' ', 'It', "'", 's', ' ', '9', ':', '30', '--', 'isn', "'", 't', ' ', 'it', '?', '']
['Hello', ',', 'Warsaw', '!', 'It', "'", 's', '9', ':', '30', '--', 'isn', "'", 't', 'it', '?']


Note what the rule does to `It's` (three tokens) and `9:30`: neither is right or wrong, but the rule has to be stated. Real tokenizers make the same kind of decision in their pre-tokenization step (Section 3.3).

### Vocabulary and lookup

The vocabulary is the sorted set of distinct tokens, with the two dictionaries `stoi` (string to integer) and `itos` (integer to string) of List 1, Materials 2.3. Decoding looks up every ID and joins the tokens.

In [3]:
corpus = "Hello, Warsaw! Hello, Wrocław!"
tokens = [t for t in re.split(r'([,.:;?_!"()\']|--|\s)', corpus) if t.strip()]
vocab = sorted(set(tokens))
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}
print(stoi)
ids = [stoi[t] for t in tokens]
print(ids)
joined = " ".join(itos[i] for i in ids)
print(repr(joined))
# remove the whitespace before punctuation
print(repr(re.sub(r'\s+([,.?!"()\'])', r"\1", joined)))

{'!': 0, ',': 1, 'Hello': 2, 'Warsaw': 3, 'Wrocław': 4}
[2, 1, 3, 0, 2, 1, 4, 0]
'Hello , Warsaw ! Hello , Wrocław !'
'Hello, Warsaw! Hello, Wrocław!'


### Decoding cannot restore what the split discarded

The split keeps the words and punctuation but throws the whitespace away: whether there was any between two tokens, which characters it was (space, tab, newline) and how many. `" ".join` must guess, and it guesses one space everywhere. `re.sub(r'\s+([,.?!"()\'])', r'\1', s)` (BLLM's fix) deletes the whitespace before punctuation: `\1` in the replacement stands for the text matched by the first group. This is right for closing marks and wrong for opening ones.

A tokenizer is **lossless** if `decode(encode(s)) == s` for every string `s`. A word tokenizer that drops whitespace cannot be lossless; it would have to store the whitespace as tokens too. Byte-level tokenizers (Section 3.4) do exactly that.

**Python you need.** `re.split` with a capturing group, `re.sub` with a group reference `r"\1"`, `str.strip`, `sorted(set(...))`, dictionary comprehensions, `" ".join(...)`; `Path("the-verdict.txt").read_text(encoding="utf-8")` as in List 1.

## 3.2 Unknown words and document boundaries

### Out-of-vocabulary words

A vocabulary built from one text cannot contain the words that only appear in another: they are **out of vocabulary (OOV)**. Without a fallback, `stoi[token]` raises a `KeyError`. BLLM's fallback is a reserved token `<|unk|>` (Lecture 3 calls it UNK) that replaces every unknown word. All unknown words get the same ID, so the tokenizer can no longer be lossless: `decode` can only print `<|unk|>`.

The **OOV rate** of a text is the share of its tokens that become `<|unk|>`. Word frequencies follow Zipf's law (Lecture 3, Part 2): a few words are very frequent and there is a long tail of rare ones, so every new text brings words never seen before, however large the vocabulary.

### Special tokens

**Special tokens** are vocabulary entries that do not stand for text but for control information (Lecture 3, "Special tokens carry instructions"):

- `<|unk|>` (BLLM's name): replaces a word that is not in the vocabulary;
- `<|endoftext|>` (GPT-2's name): marks the boundary between two documents;
- padding: fills unused positions in a batch (Lists 6–7).

Training data are usually many documents concatenated into one long token stream. `<|endoftext|>` between them tells the model that what follows does not continue what came before. Special tokens are appended to the vocabulary, so they get the last IDs. The splitting rule must keep them intact: BLLM's regular expression does not split at `<`, `|` or `>`, so `<|endoftext|>` with spaces around it stays one token.

In [4]:
specials = ["<|endoftext|>", "<|unk|>"]
itos2 = vocab + specials                       # special tokens get the last IDs
stoi2 = {tok: i for i, tok in enumerate(itos2)}
unk = stoi2["<|unk|>"]
docs = "Hello, Kraków! <|endoftext|> Hello, Warsaw!"
tokens2 = [t for t in re.split(r'([,.:;?_!"()\']|--|\s)', docs) if t.strip()]
ids = [stoi2.get(t, unk) for t in tokens2]
print(ids)
print(" ".join(itos2[i] for i in ids))

[2, 1, 6, 0, 5, 2, 1, 3, 0]
Hello , <|unk|> ! <|endoftext|> Hello , Warsaw !


`dict.get(key, default)` returns `default` for a missing key, which is all an `<|unk|>` fallback needs.

### Byte-level tokenizers need no `<|unk|>`

GPT-2 has no `<|unk|>` token. Its base vocabulary contains all 256 byte values, and every text is a sequence of bytes, so every text can be encoded (Section 3.4). It still needs `<|endoftext|>`. A new question then appears: what if a text contains the literal characters `<|endoftext|>`? A tokenizer must decide whether to treat them as the special token or as ordinary text; `tiktoken` refuses to guess and raises an error unless you say which (Section 3.4).

**Python you need.** `dict.get(key, default)`, `list.count(x)`, list concatenation `vocab + specials`, f-strings with percentages: `f"{x:.1%}"`.

## 3.3 BPE on letters

### Byte pair encoding (Lecture 3, Part 3)

Word vocabularies have OOV words; character vocabularies make sequences long. **Byte pair encoding** (BPE) learns a vocabulary in between: it starts from small units and repeatedly merges the most frequent pair of neighbouring units into a new unit.

1. Start with each word split into its smallest units, here its letters (characters).
2. Count every pair of neighbouring units in the corpus.
3. Merge the most frequent pair into a new unit and save the rule. Break ties by a fixed rule.
4. Repeat from step 2 until the vocabulary has the chosen size.

Each merge adds one vocabulary entry and shortens the corpus by the number of occurrences it replaced. The old units stay in the vocabulary as fallbacks. The lecture's worked example, with the three words `low`, `lower`, `lowest` kept as separate chunks (no merges across words) and ties broken alphabetically:

In [5]:
corpus = ["l o w", "l o w e r", "l o w e s t"]

for _ in range(3):
    counts = Counter()
    for word in corpus:
        pieces = word.split()
        counts.update(zip(pieces, pieces[1:]))        # neighbouring pairs
    pair = min(counts, key=lambda p: (-counts[p], p))  # ties: alphabetical
    corpus = [w.replace(" ".join(pair), "".join(pair)) for w in corpus]
    length = sum(len(w.split()) for w in corpus)
    print(pair, counts[pair], "-> corpus length", length)

('l', 'o') 3 -> corpus length 11
('lo', 'w') 3 -> corpus length 8
('low', 'e') 2 -> corpus length 6


`min` with the key `(-count, pair)` picks the largest count and, among equal counts, the smallest pair. (Replacing strings works for this example only; a real implementation works on lists of IDs and replaces pairs left to right, without overlaps: in `a a a`, the pair `a a` is replaced once, not twice.)

### Encoding new text applies the merges by rank

Training produces an ordered list of merges; the position of a merge in the list is its **rank**. Encoding a new text never counts anything. It starts from the smallest units and repeats: among all neighbouring pairs in the current sequence, take the pair with the **lowest rank** (learned earliest), merge all its occurrences, and stop when no pair has a merge. For the unseen word `lowers`:

$$
\texttt{l o w e r s} \;\to\; \texttt{lo w e r s} \;\to\; \texttt{low e r s} \;\to\; \texttt{lowe r s}.
$$

With the IDs $\texttt{e}, \texttt{l}, \texttt{o}, \texttt{r}, \texttt{s}, \texttt{t}, \texttt{w} = 0, \dots, 6$, $\texttt{lo} = 7$, $\texttt{low} = 8$ and $\texttt{lowe} = 9$, `lowers` encodes to `[9, 3, 4]`. Decoding looks up the piece of each ID and concatenates: `lowe` + `r` + `s`.

### Chunks: no merges across words

Before training, the text is cut into **chunks**, and merges never cross a chunk boundary. A simple splitting rule with `re.findall` (it returns all non-overlapping matches, left to right):

```python
r" ?\w+| ?[^\w\s]+|\s+"
```

Each match is a word with at most one space in front (`\w`: a letter, digit or underscore, in any script), a run of punctuation with at most one space in front, or a run of whitespace. Every character falls into one of these classes, so joining the chunks gives back the text exactly.

In [6]:
CHUNK = r" ?\w+| ?[^\w\s]+|\s+"
s = "Hello, Warsaw!  It's 2026."
chunks = re.findall(CHUNK, s)
print(chunks)
print("".join(chunks) == s)
print(Counter(re.findall(CHUNK, "the cat and the dog and the bird")))

['Hello', ',', ' Warsaw', '!', '  ', 'It', "'", 's', ' 2026', '.']
True
Counter({' and': 2, ' the': 2, 'the': 1, ' cat': 1, ' dog': 1, ' bird': 1})


The space belongs to the following word, as in GPT-2 (Section 3.4): ` Warsaw` (with its leading space) and `Warsaw` are different chunks. The second line counts the chunks of a short text: identical chunks always receive identical merges, which makes training faster (below).

Chunks matter. If `low lower lowest` is trained as one string with spaces, the first two merges are the same, but then the pair (space, `low`) also occurs twice, ties with (`low`, `e`), and wins, because the space sorts before every letter: the third merge becomes ` low`. Without chunks, frequent phrases such as `of the` become tokens.

### The base vocabulary is the alphabet of the training text

The base vocabulary of letter-level BPE is the set of characters in the training text, `sorted(set(text))`, with IDs $0, \dots, A - 1$ in sorted order; merged tokens get the IDs $A, A + 1, \dots$. Ties between pairs with equal counts are then broken by the smaller pair of IDs, which for single letters is alphabetical order. A character that does not occur in the training text has no ID and cannot be encoded. This is the OOV problem of Section 3.2 again, one level lower, and Section 3.4 solves it.

### Making training fast enough

Each merge needs a fresh count of all pairs, so training costs about (number of merges) × (corpus length) steps. For a text of 20,000 characters and a few hundred merges this takes seconds in plain Python. Many chunks are identical (` the` occurs hundreds of times): count each distinct chunk once, weighted by its frequency (`Counter(chunks)`), and apply each merge to every distinct chunk once.

**Python you need.** `re.findall`; `sorted(set(text))`; `collections.Counter` (`update`, `+=`); `zip(ids, ids[1:])` for neighbouring pairs; `min(..., key=...)`; `dict.get(pair, math.inf)` for the rank of a pair without a merge; `try` / `except KeyError`; `plt.plot`.

## 3.4 From letters to bytes: byte-level BPE

### Bytes, Unicode and UTF-8 (Lecture 3, Part 4)

An alphabet of the characters seen in training cannot spell `Łódź` if it never saw `Ł`. Computers store all text as **bytes** (8 bits, 256 possible values), which gives a fixed alphabet for every writing system:

- **Unicode** gives every character a number, its **code point**: `ord("é")` is 233.
- **UTF-8** stores each code point as 1 to 4 bytes: ASCII characters (English letters, digits, punctuation) as one byte with the same value, others as several bytes, each between 128 and 255. `é` becomes the two bytes 195, 169.

In Python, `s.encode("utf-8")` turns a string into a `bytes` object, `list(b)` gives the byte values as integers, and `bytes(ints).decode("utf-8")` goes back.

In [7]:
for s in ["A", "é", "猫", "Zażółć gęślą jaźń"]:
    b = s.encode("utf-8")
    print(f"{s!r:20s} {len(s):2d} characters, {len(b):2d} bytes: {list(b)[:8]}")
print(bytes([195, 169]).decode("utf-8"))

'A'                   1 characters,  1 bytes: [65]
'é'                   1 characters,  2 bytes: [195, 169]
'猫'                   1 characters,  3 bytes: [231, 140, 171]
'Zażółć gęślą jaźń'  17 characters, 26 bytes: [90, 97, 197, 188, 195, 179, 197, 130]
é


Note that `len(s)` counts characters, not bytes: the two agree only for ASCII text.

### Byte-level BPE

Byte-level BPE runs the algorithm of Section 3.3 on bytes instead of letters. Only the base vocabulary changes:

- the **base vocabulary** is the 256 byte values, with IDs $0, \dots, 255$; the first merge gets ID 256, the next 257, and so on;
- a chunk is encoded by turning it into its UTF-8 bytes and applying the merges by rank;
- a tie between pairs with equal counts is broken by the smaller pair of IDs. For ASCII characters the byte value equals the code point, so on ASCII text the order, the counts and hence the merges are the same as with letters: `l` + `o` = (108, 111) comes before `o` + `w` = (111, 119).

Because the 256 base tokens are never removed, **every** string can be encoded, and there are no OOV tokens: an unknown word, or an unknown script, just takes more, shorter tokens. Merges only add shortcuts.

### Decoding: concatenate the bytes, then decode UTF-8

The bytes of a merged token are the bytes of its two parts, concatenated; building a table `ID → bytes` in merge order gives every token's bytes. Decoding joins the bytes of all tokens with `b"".join(...)` and decodes the result once. Decoding each token on its own is wrong: a token can contain part of a multi-byte character (Lecture 3, "Lossless decoding joins bytes before decoding UTF-8").

In [8]:
pieces = [bytes([195]), bytes([169])]   # the two bytes of "é" as two tokens
print(b"".join(pieces).decode("utf-8"))  # join first: correct
# token by token: replacement characters
print([p.decode("utf-8", errors="replace") for p in pieces])
try:
    pieces[0].decode("utf-8")
except UnicodeDecodeError as e:
    print("strict:", e)

é
['�', '�']
strict: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data


`errors="replace"` puts the replacement character `�` where the bytes are not valid UTF-8, instead of raising an error.

### GPT-2: pre-tokenization and the byte map

GPT-2 (Radford et al., 2019) also cuts the text into chunks with a regular expression, its **pre-tokenization**, and applies byte-level BPE inside each chunk. The next cell contains the pattern from its [reference implementation](https://github.com/openai/gpt-2/blob/master/src/encoder.py), split over two lines. Read the alternatives left to right: common English contractions; a run of letters (`\p{L}`, any script) with at most one space in front; a run of digits (`\p{N}`); a run of other characters; and runs of whitespace. `\p{L}` is not supported by Python's `re`; the `regex` module (installed with `tiktoken`) supports it. The rule of Section 3.3 is a simplified version: it keeps letters and digits together and has no special cases for contractions.

In [9]:
import regex

GPT2_SPLIT = (r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+"""
              r"""| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
print(regex.findall(GPT2_SPLIT, "Hello, Warsaw!  It's 2026."))

['Hello', ',', ' Warsaw', '!', ' ', ' It', "'s", ' 2026', '.']


GPT-2's files store each token as a printable string, with every byte mapped to a visible Unicode character (the space byte appears as `Ġ`). This **byte map** is only a storage convention; you will see it when you print GPT-2's vocabulary, never in the model's input. GPT-2's vocabulary has $50{,}257$ entries: 256 bytes, 50,000 merges and `<|endoftext|>` with ID 50256.

### `tiktoken`

`tiktoken` implements GPT-2's tokenizer (and newer ones) in fast compiled code:

In [10]:
enc = tiktoken.get_encoding("gpt2")   # downloads the files on the first call
ids = enc.encode("Hello, Warsaw!")
print(ids, "->", repr(enc.decode(ids)), "| vocabulary size:", enc.n_vocab)
print([enc.decode_single_token_bytes(t) for t in ids])
print(enc.encode("Bye.<|endoftext|>Hi.", allowed_special={"<|endoftext|>"}))
try:
    enc.encode("Bye.<|endoftext|>Hi.")
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")

[15496, 11, 32955, 0] -> 'Hello, Warsaw!' | vocabulary size: 50257
[b'Hello', b',', b' Warsaw', b'!']
[3886, 68, 13, 50256, 17250, 13]
ValueError: Encountered text corresponding to disallowed special token '<|endoftex ...


`enc.encode` raises an error when the text contains a special-token string, unless you allow it with `allowed_special` (it becomes the special token, ID 50256) or use `enc.encode_ordinary`, which encodes it as plain text.

### How multi-byte characters get merged

Byte-level BPE has no notion of a character; it only sees bytes ([Lecture 3, Q&A](../../lectures/03-tokenization/QA.md), Question 1):

1. **At the start every byte is a token.** `ą` (U+0105) is the two bytes `C4 85`, `中` the three bytes `E4 B8 AD`, `😀` the four bytes `F0 9F 98 80`.
2. **Merges are chosen by frequency only.** The bytes of a character that is frequent in the training data are merged into one token early; those of a rare character may never be merged.
3. **Merges ignore character boundaries inside a chunk.** A token can hold the last byte of one character and the first byte of the next, or a space and the first byte of a character. Such a token is not valid UTF-8 on its own.
4. **Merges never cross chunk boundaries.** Polish letters count as letters (`\w`, `\p{L}`), so `ą` stays in the chunk of its word. Chinese is written without spaces, so a whole run of Chinese characters is one chunk, and merges inside it can cross from one character to the next.

The value of a byte tells where it sits in its character:

| byte value (hex) | role in UTF-8 |
|---|---|
| `00`–`7F` | a whole ASCII character |
| `80`–`BF` | continuation byte: 2nd, 3rd or 4th byte of a character |
| `C2`–`F4` | lead byte: 1st byte of a 2-, 3- or 4-byte character |

A token decodes with strict UTF-8 (`token.decode("utf-8")` without `errors=`) only if it consists of whole characters. If it does not, the byte roles show whether it is part of one character or crosses a character boundary.

In [11]:
def byte_role(b):
    if b < 0x80:
        return "ASCII"
    return "continuation" if b < 0xC0 else "lead"


for ch in "aą中😀":
    bs = ch.encode("utf-8")
    print(f"{ch!r}: {len(bs)} bytes", [f"{b:02X} {byte_role(b)}" for b in bs])

'a': 1 bytes ['61 ASCII']
'ą': 2 bytes ['C4 lead', '85 continuation']
'中': 3 bytes ['E4 lead', 'B8 continuation', 'AD continuation']
'😀': 4 bytes ['F0 lead', '9F continuation', '98 continuation', '80 continuation']


GPT-2 shows all four effects. Its tokens for a few words, as bytes in hex:

In [12]:
for s in [" Wrocław", " Łódź", "ą", "中", "文", "😀"]:
    tokens = [enc.decode_single_token_bytes(t) for t in enc.encode(s)]
    print(f"{s!r:10s}", [t.hex(" ") for t in tokens])

' Wrocław' ['20 57', '72 6f 63', 'c5 82', '61 77']
' Łódź'    ['20 c5', '81', 'c3 b3', '64', 'c5', 'ba']
'ą'        ['c4', '85']
'中'        ['e4 b8 ad']
'文'        ['e6 96', '87']
'😀'        ['f0 9f 98', '80']


`ł` (`C5 82`) is frequent enough to be one token, `Ł` and `ź` are not: in ` Łódź` the first byte of `Ł` is merged with the preceding space, and the two bytes of `ź` are separate tokens. `中` is one token, `文` is split into two, and `😀` into two tokens, neither of them a valid character.

**Consequence.** Text in languages whose characters take several UTF-8 bytes, and which were rare in the tokenizer's training data, needs more tokens for the same content: less text fits in the context window, and processing costs more per word. Coverage is never the problem, because every byte is in the vocabulary.

### Measuring compression

**Bytes per token**, $M/N$ for a text of $M$ UTF-8 bytes and $N$ tokens, measures how much text a token carries on average. Byte-level BPE gives frequent sequences of its training data long tokens; text that differs from the training data (another language, script or domain) is split into shorter pieces, down to single bytes (Lecture 3, "Byte coverage does not imply equal efficiency").

**Python you need.** `str.encode("utf-8")`, `bytes.decode("utf-8")` and `errors="replace"`, `list(b)`, `bytes([...])`, `b"".join(...)`, `bytes.hex(" ")`, `bytes.isascii()`; `try` / `except UnicodeDecodeError`; `tiktoken.get_encoding`, `encode`, `decode`, `decode_single_token_bytes`, `n_vocab`, `allowed_special`; your functions from Task 3.3.

## 3.5 Comparing tokenizers with bits per byte

### Units of the loss

The loss of a language model on a token is its **negative log-likelihood** (NLL), $-\ln p$, where $p$ is the probability the model gave to the correct token (List 1, Materials 2.2–2.3). With the natural logarithm the unit is the **nat**; dividing by $\ln 2$ converts to **bits**:

| probability of the correct token | loss in nats | loss in bits |
|---|---:|---:|
| $1/2$ | 0.693 | 1 |
| $1/4$ | 1.386 | 2 |

### Perplexity depends on the tokenizer

Perplexity is $\exp$ of the mean NLL **per token**. Two tokenizers cut the same text into different numbers of tokens, and a small unit (a byte) is easier to predict than a large one (a word), so per-token perplexities of different tokenizers are not comparable. Lecture 3's example: the same 100-byte text, system A with 50 tokens and mean NLL 1.0 nats, system B with 25 tokens and mean NLL 1.5 nats. A has the lower perplexity, but B spends fewer nats on the whole text.

### Bits per byte

Divide the **total** NLL by a unit that does not depend on the tokenizer, the number of bytes of the text:

$$
\mathrm{BPB} = \frac{-\sum_{t=1}^{N} \ln p_\theta(z_t \mid z_{<t})}{M \ln 2} = \frac{N \bar\ell}{M \ln 2},
$$

where $z_1, \dots, z_N$ are the $N$ scored tokens of the encoded text, $\bar\ell$ is their mean NLL in nats per token, $M$ is the number of UTF-8 bytes of the same text, and $\theta$ are the model's parameters. Lower is better. BPB is the number of bits per byte an ideal compressor driven by the model would need, so it compares **model + tokenizer** systems on the same text.

In [13]:
num_bytes = 100
for name, num_tokens, mean_nll in [("A", 50, 1.0), ("B", 25, 1.5)]:
    bpb = num_tokens * mean_nll / (num_bytes * math.log(2))
    print(f"{name}: perplexity {math.exp(mean_nll):.2f}, BPB {bpb:.3f}")

A: perplexity 2.72, BPB 0.721
B: perplexity 4.48, BPB 0.541


### Scoring held-out text with a bigram model

The add-$\alpha$ bigram model of List 1, Task 2.3, works for any token IDs: count the pairs with `torch.bincount(ids[:-1] * V + ids[1:], minlength=V * V)`, add $\alpha$, normalise the rows. The table has $V^2$ entries, a million for $V = 1024$, and a text of a few thousand tokens fills very few of them, so the smoothing matters more as $V$ grows.

Three details make BPB comparable across tokenizers:

- Use the **same held-out text** and the same $M$ for every tokenizer; only the encoding differs.
- Score **every** held-out token, including the first: use the last training token as its context. Otherwise one tokenizer skips a byte and another skips two or three.
- Sum the NLL over all tokens; do not average averages.

Training to vocabulary size $V$ gives exactly the first $V - 256$ merges of a longer run, because each merge depends only on the earlier ones. To compare several sizes, train once to the largest and keep the merges with IDs below $V$.

**Python you need.** `math.log(2)`, `math.exp`, `torch.bincount(..., minlength=...)`, `.view(V, V)`, advanced indexing `P[a, b]`, `torch.cat`, `.sum()`, `.mean()`; your bigram code from List 1.

## 3.6 Tokenization consequences

A model sees token IDs, not letters or digits (Lecture 3, Part 7):

- **Spelling and counting.** A word that is one or two tokens gives the model no direct access to its letters. Counting letters requires knowledge about the spelling of each token, learned indirectly from data.
- **Arithmetic.** Numbers are split into chunks whose boundaries depend on the digits, so the columns that written addition works on (units, tens, hundreds) do not line up with the token boundaries.
- **Whitespace.** A leading space is part of a GPT-2 token, so `Warsaw` and ` Warsaw` (with a leading space) are different tokens, and extra spaces produce different IDs.

Tokenization adds an obstacle; it does not make these tasks impossible, and it is not the only cause of such errors. Inspect the actual tokens before blaming it:

In [14]:
for s in ["Mississippi", " Mississippi", "3.14159", "Warsaw", " Warsaw"]:
    pieces = [enc.decode_single_token_bytes(t) for t in enc.encode(s)]
    print(f"{s!r:16s}", [p.decode("utf-8") for p in pieces])

'Mississippi'    ['Miss', 'iss', 'ippi']
' Mississippi'   [' Mississippi']


'3.14159'        ['3', '.', '14', '159']
'Warsaw'         ['Wars', 'aw']
' Warsaw'        [' Warsaw']


**Python you need.** `enc.encode`, `enc.decode_single_token_bytes(t).decode("utf-8")`, `repr`.

---

# Lab 4: Sampling & Embedding Pipeline

## 4.1 An embedding is a one-hot vector times a matrix

### Why vectors

Token IDs are labels: ID 5 is not closer to ID 6 than to ID 500, and arithmetic on IDs means nothing. A model needs a representation in which related tokens can be close, so every token gets a learned vector in $\mathbb{R}^d$, its **embedding**. The **distributional hypothesis** (Harris, 1954; "You shall know a word by the company it keeps", Firth, 1957) says that words occurring in similar contexts have similar meanings. Training a model to predict context therefore pushes the embeddings of such words together: explicitly in word2vec (Section 4.5), and implicitly in a GPT, whose embeddings are learned with the rest of the model by next-token prediction.

### The embedding matrix

All embeddings form one matrix $E \in \mathbb{R}^{V \times d}$, one row per token. With the one-hot vector $\mathbf{e}_i \in \mathbb{R}^V$ (1 at position $i$, 0 elsewhere),

$$
\text{embedding}(i) = E_{i,:} = \mathbf{e}_i^\top E .
$$

For a sequence of $T$ IDs, stacking the one-hot vectors as rows of $X \in \{0, 1\}^{T \times V}$ gives all embeddings at once: $XE \in \mathbb{R}^{T \times d}$. List 1, Materials 2.4, checked this for `nn.Embedding`; the lookup gives the same result as the product without ever forming $X$.

### `nn.Embedding` and `nn.Linear` store the matrix differently

`nn.Linear(d_in, d_out)` computes $\mathbf{x} W^\top$ and stores `weight` with shape $(d_{\text{out}}, d_{\text{in}})$ (List 1, Materials 1.4). So `nn.Linear(V, d, bias=False)` applied to one-hot rows computes $X W^\top$ with $W \in \mathbb{R}^{d \times V}$, and it equals the embedding lookup when $W = E^\top$.

In [15]:
emb = nn.Embedding(6, 3)            # E: (V, d) = (6, 3)
lin = nn.Linear(6, 3, bias=False)   # W: (d, V) = (3, 6)
print(emb.weight.shape, lin.weight.shape)
with torch.no_grad():               # the copy is not recorded by autograd
    lin.weight.copy_(emb.weight.T)  # W = E^T
print(torch.equal(lin.weight, emb.weight.T))

torch.Size([6, 3]) torch.Size([3, 6])
True


### Sparse gradients

For $\text{out} = XE$ the chain rule gives

$$
\frac{\partial L}{\partial E} = X^\top \frac{\partial L}{\partial\, \text{out}} \in \mathbb{R}^{V \times d} .
$$

Row $j$ of $X^\top$ has ones exactly at the positions where token $j$ occurs. Work out what this means for the rows of the gradient of tokens that occur twice, once, or not at all in the batch.

### Cost

The lookup copies $T$ rows: $O(Td)$ work. The product with one-hot vectors builds a $T \times V$ matrix and costs $O(TVd)$. For GPT-2 ($V = 50{,}257$) the difference is large, which is why every model uses the lookup.

**PyTorch you need.** `nn.Embedding`, `nn.Linear(..., bias=False)`, `F.one_hot(idx, num_classes=V).float()`, `.weight`, `.T`, `torch.no_grad()`, `.copy_`, `torch.allclose`, `.backward()`, `.grad`, `.nonzero()`; `time.perf_counter()` for timing.

## 4.2 Sliding-window samples

### Inputs and targets for next-token prediction

A language model predicts each token from the ones before it (List 1, Materials 2.3). From a token sequence $x_0, \dots, x_{N-1}$, a **window** of length $L$ starting at position $i$ gives one training example:

$$
\text{input} = (x_i, \dots, x_{i+L-1}), \qquad \text{target} = (x_{i+1}, \dots, x_{i+L}) .
$$

The target is the input shifted by one token. One window contains $L$ predictions: the target at position $j$ is to be predicted from inputs $0, \dots, j$. (That the model looks only at those inputs is ensured later by the causal mask of List 3.)

### Stride and overlap

Windows start at $i = 0, s, 2s, \dots$; the **stride** $s$ is the distance between two starts.

- $s < L$: consecutive windows overlap by $L - s$ tokens. The same target appears in several windows with different contexts: more examples, but strongly correlated ones.
- $s = L$: the windows tile the text without overlap, and every token is a target exactly once (except $x_0$ and the few tokens after the last full window).
- $s > L$: tokens between windows are never used.

In [16]:
ids = list(range(10, 20))           # stand-ins for 10 token IDs
L, s = 4, 3
for i in range(0, len(ids) - L, s):
    print(f"start {i}: input {ids[i : i + L]}  target {ids[i + 1 : i + L + 1]}")

start 0: input [10, 11, 12, 13]  target [11, 12, 13, 14]
start 3: input [13, 14, 15, 16]  target [14, 15, 16, 17]


### Counting windows

The window starting at $i$ needs the tokens up to $x_{i+L}$, its last target, so it exists only if $i + L + 1 \le N$, that is $i \le N - L - 1$. The loop `range(0, N - L, s)` produces exactly these starts. Counting the multiples of $s$ in $[0, N - L - 1]$ is Task 4.2.

### A `Dataset` of windows

The `Dataset` / `DataLoader` split of List 1, Materials 2.1, fits directly: the dataset tokenizes the text once, computes all windows in `__init__`, and `__getitem__(j)` returns window $j$ as a pair `(input, target)` of `int64` tensors of shape $(L,)$. The `DataLoader` stacks them into batches of shape $(B, L)$. `create_dataloader` wraps both steps; its exact specification is in Appendix A. The token IDs come from GPT-2's tokenizer: `tiktoken.get_encoding("gpt2").encode(text, allowed_special={"<|endoftext|>"})`, so that texts joined by `<|endoftext|>` work (Section 3.4).

`drop_last=True` skips an incomplete last batch, so every training batch has the same shape; `shuffle=True` is for training data only. Keep `num_workers=0` in notebooks.

**PyTorch you need.** `Dataset` (`__init__`, `__len__`, `__getitem__`), `DataLoader(dataset, batch_size=..., shuffle=..., drop_last=..., num_workers=...)`, `torch.tensor(list_of_ints)`, `loader.dataset`, `len(...)`, `next(iter(loader))`; `enc.decode([t])` to print a token.

## 4.3 Token and positional embeddings

### Why positions are needed

Self-attention (List 3) computes each output vector as a weighted average of all input vectors, with weights computed from the vectors themselves. Nothing in it depends on *where* a vector sits in the sequence. As a consequence it is **permutation-equivariant**: permuting the input vectors permutes the output vectors in the same way, $f(PX) = P f(X)$ for every permutation matrix $P$. The token embedding is applied to each position separately and is equivariant too. So `dog bites man` and `man bites dog` would reach the model as the same set of vectors. To make the order visible, information about the position is added to the input vectors.

### Learned absolute positions (GPT-2)

GPT-2 uses a second embedding table $P \in \mathbb{R}^{T_{\max} \times d}$ with one row per position $0, \dots, T_{\max} - 1$, where $T_{\max}$ is the **context length**. The input vector at position $t$ is

$$
\mathbf{h}_t = E_{x_t,:} + P_{t,:} .
$$

$P$ is learned with the rest of the model, like $E$. It has no row for positions $\ge T_{\max}$, so a model with learned absolute positions cannot read longer sequences.

### Broadcasting

For a batch of IDs of shape $(B, T)$, the token embeddings have shape $(B, T, d)$ and the position embeddings of positions $0, \dots, T - 1$ shape $(T, d)$. Adding them broadcasts over the batch (List 1, Materials 1.1): every sequence gets the same position vectors. Create the positions with `torch.arange(T, device=idx.device)`, so the code also works on a GPU.

In [17]:
B, T, d = 2, 3, 4
tok = torch.zeros(B, T, d)                     # stand-in: token embeddings
pos = torch.arange(T * d, dtype=torch.float32).view(T, d)  # stand-in: positions
h = tok + pos
print(h.shape)                                 # (2, 3, 4)
print(torch.equal(h[0], h[1]))                 # same positions in each row

torch.Size([2, 3, 4])
True


### GPT-2 sizes

All GPT-2 models (Radford et al., 2019, [paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), Table 2) use the vocabulary of $V = 50{,}257$ tokens and context length $T_{\max} = 1024$; they differ in the embedding dimension:

| model | $d$ | total parameters |
|---|---:|---:|
| GPT-2 small | 768 | 124M |
| GPT-2 medium | 1024 | 355M |
| GPT-2 large | 1280 | 774M |
| GPT-2 XL | 1600 | 1558M |

The two embedding tables have $(V + T_{\max})\, d$ parameters. (The paper reports 117M, 345M, 762M and 1542M; the table gives the counts of the released weights, which BLLM uses.)

**PyTorch you need.** `nn.Module`, `nn.Embedding`, `torch.arange(T, device=...)`, broadcasting, `idx.shape`, indexing a permutation `idx[:, perm]`, `torch.equal`, `torch.allclose`; `count_parameters` from List 1.

## 4.4 Sinusoidal positional encodings

### A fixed function of the position

Vaswani et al. (2017) use no learned table. The encoding of position $t$ is a fixed vector $\mathrm{PE}(t) \in \mathbb{R}^d$ ($d$ even) whose entries come in pairs $i = 0, \dots, d/2 - 1$:

$$
\mathrm{PE}(t)_{2i} = \sin(\omega_i t), \qquad
\mathrm{PE}(t)_{2i+1} = \cos(\omega_i t), \qquad
\omega_i = 10000^{-2i/d} .
$$

- $t = 0, 1, 2, \dots$ is the position;
- $i$ numbers the (sin, cos) pairs: even dimensions hold sines, odd dimensions cosines;
- $\omega_i$ is the frequency of pair $i$ in radians per position, from $\omega_0 = 1$ down to almost $10^{-4}$; its wavelength $2\pi/\omega_i$ goes from $2\pi \approx 6.3$ to almost $2\pi \cdot 10^4$ positions.

Each pair is a point on the unit circle that turns by the angle $\omega_i$ per position: the first pairs turn fast and distinguish neighbouring positions, the last ones turn slowly and distinguish distant ones, like the hands of a clock. The encoding is added to the token embeddings just like the learned one, and it works for any position.

In [18]:
d = 8
i = torch.arange(d // 2)
omega = 10000.0 ** (-2 * i / d)                  # (d/2,)
print("omega_i:     ", [f"{w:.4f}" for w in omega.tolist()])
print("wavelengths: ", [f"{2 * math.pi / w:.0f}" for w in omega.tolist()])

omega_i:      ['1.0000', '0.1000', '0.0100', '0.0010']
wavelengths:  ['6', '63', '628', '6283']


`torch.arange(0, d, 2)` gives the even indices $2i$ directly. To build all of $\mathrm{PE}$ at once, multiply a column of positions of shape $(T, 1)$ by the row of frequencies of shape $(d/2,)$: broadcasting gives all angles $\omega_i t$ as a $(T, d/2)$ matrix. Then fill the even columns with `pe[:, 0::2] = ...` and the odd ones with `pe[:, 1::2] = ...`.

### Dot products depend only on the offset

For one (sin, cos) pair, the product of the encodings at positions $t$ and $t + k$ is $\sin(\omega_i t)\sin(\omega_i(t+k)) + \cos(\omega_i t)\cos(\omega_i(t+k))$. The identity $\cos(a - b) = \cos a \cos b + \sin a \sin b$ simplifies it; sum over the pairs to get $\mathrm{PE}(t)\cdot\mathrm{PE}(t+k)$.

### Shifting by $k$ is a rotation

The angle-addition formulas

$$
\sin(a + b) = \sin a \cos b + \cos a \sin b, \qquad
\cos(a + b) = \cos a \cos b - \sin a \sin b
$$

with $a = \omega_i t$ and $b = \omega_i k$ express the pair $(\sin \omega_i(t+k), \cos \omega_i(t+k))$ as a $2 \times 2$ matrix that depends only on $\omega_i k$, times the pair $(\sin \omega_i t, \cos \omega_i t)$. Placing the $d/2$ blocks on the diagonal of a $d \times d$ matrix gives $R(k)$, with $\mathrm{PE}(t+k) = R(k)\,\mathrm{PE}(t)$ for every $t$: a shift in position is a linear map that does not depend on where you start. `torch.block_diag(*blocks)` builds a block-diagonal matrix from a list of blocks. With positions stored as rows of a $(T, d)$ matrix, $R(k)\,\mathrm{PE}(t)$ for all $t$ at once is `pe @ R.T`. Check in `float64`: in `float32` the errors are around $10^{-6}$ and hide whether the formula is exact.

### Preview: rotary position embeddings

**RoPE** (Su et al., 2021, [paper](https://arxiv.org/abs/2104.09864)), used in many recent LLMs such as Llama, turns this property into the mechanism. Instead of adding an encoding to the input, it rotates the query and key vectors of attention by $R(t)$ at their position $t$. Since rotations satisfy $R(t)^\top R(s) = R(s - t)$, the attention score $(R(t)\mathbf{q})^\top (R(s)\mathbf{k}) = \mathbf{q}^\top R(s - t)\,\mathbf{k}$ depends only on the offset $s - t$.

**PyTorch you need.** `torch.arange(..., dtype=torch.float64)`, `torch.sin`, `torch.cos`, broadcasting `(T, 1) * (d/2,)`, slicing `pe[:, 0::2]`, `torch.block_diag`, `@`, `.T`, `torch.allclose`, `.abs().max()`; `plt.imshow(pe, aspect="auto")`, `plt.colorbar()`, `plt.subplots`.

## 4.5 Skip-gram with negative sampling *(optional)*

### The skip-gram model

word2vec (Mikolov et al., 2013, [paper](https://arxiv.org/abs/1301.3781)) learns word embeddings by predicting context. For every word $w$ in a text and every word $o$ at most $c$ positions away (the **window**, typically $c = 2$ to 5), the pair $(w, o)$ is a training example. Each word has two vectors: $\mathbf{u}_w$ when it is the centre word and $\mathbf{v}_w$ when it is a context word. Predicting $o$ from $w$ with a softmax over the whole vocabulary costs $O(V)$ per example.

### Negative sampling

**Negative sampling** (Mikolov et al., 2013, [paper](https://arxiv.org/abs/1310.4546)) replaces the softmax by a binary task: tell the true pair $(w, o)$ from $K$ random pairs $(w, n_1), \dots, (w, n_K)$, with the noise words drawn from $P_n(n) \propto \text{count}(n)^{3/4}$. The loss of one example is

$$
\ell(w, o) = -\log \sigma(\mathbf{u}_w^\top \mathbf{v}_o) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_w^\top \mathbf{v}_{n_k}),
$$

logistic regression (List 1, Materials 1.2) with the dot product as the score: large for observed pairs, small for random ones. Typical choices: $K = 5$ to 20, $d = 50$ to 300. After training, the $\mathbf{u}_w$ are the embeddings.

### Nearest neighbours

Similarity of two embeddings is measured by the **cosine similarity** $\cos(\mathbf{a}, \mathbf{b}) = \mathbf{a}^\top \mathbf{b} / (\lVert \mathbf{a} \rVert \lVert \mathbf{b} \rVert)$; with all rows normalised to length 1 (`F.normalize(E, dim=1)`), the similarities of one word to all others are a single product `E @ E[i]`. word2vec was trained on billions of words; on a small text expect much noisier neighbours.

**PyTorch you need.** two `nn.Embedding`s, `F.logsigmoid`, `torch.multinomial(weights, n, replacement=True)`, products and `.sum(-1)` along the last dimension, `F.normalize`, `.topk`.

## 4.6 Ideas for your own experiment *(optional)*

About one hour of work; one plot or table and a paragraph is enough.

- Train your BPE on a Polish text (for example a book from [Wolne Lektury](https://wolnelektury.pl/)) and on the English default text. Compare bytes per token of both tokenizers on both languages.
- Plot BPB of the bigram model against the number of BPE merges. Where is the best vocabulary size, and does it move when the training text gets longer?
- Compare GPT-2's tokenizer with a newer one (`tiktoken.get_encoding("cl100k_base")`, a one-time download) on numbers, code and Polish text.
- How does the base 10000 of the sinusoidal encoding change the curve of $\mathrm{PE}(t) \cdot \mathrm{PE}(t+k)$?
- In skip-gram, compare window sizes $c = 1$ and $c = 5$. Do the neighbours change character (grammatical against topical)?
- Implement WordPiece's longest-match encoding (Lecture 3, Part 5) with your BPE vocabulary and compare the segmentations of a few words with BPE's.

---

## Appendix A: The two reusable definitions

`create_dataloader` (Task 4.2) and `GPTEmbedding` (Task 4.3) are used again in Lists 4–5, so their signatures are fixed. Full specification:

```python
def create_dataloader(
    text: str,
    batch_size: int = 4,
    max_length: int = 256,
    stride: int = 128,
    shuffle: bool = True,
    drop_last: bool = True,
    num_workers: int = 0,
) -> DataLoader:
    """Data loader of next-token prediction windows over a text.

    Encodes `text` with tiktoken's GPT-2 encoding, allowing the special
    token "<|endoftext|>", into token IDs x_0, ..., x_{N-1}. Window j
    starts at token i = j * stride and exists while
    i + max_length + 1 <= N, so there are
    (N - max_length - 1) // stride + 1 windows
    (the starts of range(0, N - max_length, stride)).

    loader.dataset[j] is a pair (input, target) of int64 tensors of
    shape (max_length,):
        input  = x_i, ..., x_{i + max_length - 1}
        target = x_{i + 1}, ..., x_{i + max_length}
    The loader stacks them into batches of shape
    (batch_size, max_length); batch_size, shuffle, drop_last and
    num_workers are passed on to DataLoader.

    Returns:
        A DataLoader yielding (inputs, targets).
    """
    ...


class GPTEmbedding(nn.Module):
    """Token embedding plus learned absolute position embedding.

    Attributes:
        tok_emb: nn.Embedding(vocab_size, emb_dim), one row per
                 token ID.
        pos_emb: nn.Embedding(context_length, emb_dim), one row per
                 position 0, ..., context_length - 1.
    """

    def __init__(self, vocab_size: int, context_length: int,
                 emb_dim: int):
        ...

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """(B, T) int64 token IDs, T <= context_length, to
        tok_emb(idx) + pos_emb(positions 0, ..., T - 1),
        of shape (B, T, emb_dim)."""
        ...
```

To check your implementations, run this after defining them:

```python
_text = ("Hello, Warsaw! " * 30 + "<|endoftext|>"
         + " Zażółć gęślą jaźń." * 10)
_enc = tiktoken.get_encoding("gpt2")
_ids = _enc.encode(_text, allowed_special={"<|endoftext|>"})
_N = len(_ids)
for _L, _s in [(4, 4), (8, 3), (16, 16)]:
    _loader = create_dataloader(_text, batch_size=2, max_length=_L,
                                stride=_s, shuffle=False,
                                drop_last=False)
    assert len(_loader.dataset) == (_N - _L - 1) // _s + 1, "windows"
    _x, _y = _loader.dataset[1]
    assert _x.dtype == torch.long, "token IDs must be int64"
    assert _x.tolist() == _ids[_s : _s + _L], "window 1 starts at s"
    assert _y.tolist() == _ids[_s + 1 : _s + _L + 1], "shift by one"
    _xb, _yb = next(iter(_loader))
    assert _xb.shape == _yb.shape == (2, _L)
_loader = create_dataloader(_text, batch_size=4, max_length=4,
                            stride=4)
assert len(_loader) == ((_N - 5) // 4 + 1) // 4, "drop_last"

_emb = GPTEmbedding(vocab_size=100, context_length=8, emb_dim=16)
assert isinstance(_emb.tok_emb, nn.Embedding)
assert isinstance(_emb.pos_emb, nn.Embedding)
assert tuple(_emb.tok_emb.weight.shape) == (100, 16)
assert tuple(_emb.pos_emb.weight.shape) == (8, 16)
assert sum(p.numel() for p in _emb.parameters()) == 100 * 16 + 8 * 16
_gen = torch.Generator().manual_seed(0)
_idx = torch.randint(0, 100, (3, 5), generator=_gen)
_out = _emb(_idx)
assert _out.shape == (3, 5, 16)
_pos = _emb.pos_emb.weight[:5]
assert torch.allclose(_out, _emb.tok_emb(_idx) + _pos)
_full = torch.zeros(1, 8, dtype=torch.long)
assert _emb(_full).shape == (1, 8, 16), "T = context_length must work"
print("Contract check passed: create_dataloader, GPTEmbedding")
```

## Appendix B: Checkpoints

**Lab 3.** Your byte-level BPE must reproduce every string exactly. Pass your own `encode` and `decode` (functions of one argument) to `check_roundtrip`; for example, with a trained `merges` table:

```python
def check_roundtrip(encode, decode):
    strings = ["", " leading space", "repeated   spaces",
               "two\nlines\n\n", "Zażółć gęślą jaźń",
               "猫が好きです", "👍🏽 thumbs up", "<|endoftext|>"]
    for s in strings:
        assert decode(encode(s)) == s, f"round trip fails for {s!r}"
    print("Checkpoint Lab 3 passed")


# pass your own encode and decode, for example:
check_roundtrip(lambda s: bpe_encode(s, merges),
                lambda ids: bpe_decode(ids, merges))
```

**Lab 4.** With `text` the default text and your two definitions from Tasks 4.2 and 4.3:

```python
loader = create_dataloader(text, batch_size=8, max_length=4, stride=4,
                           shuffle=False)
inputs, targets = next(iter(loader))
emb = GPTEmbedding(50257, 4, 256)
assert inputs.shape == targets.shape == (8, 4)
assert inputs.dtype == torch.long
assert torch.equal(targets[:, :-1], inputs[:, 1:])
assert emb(inputs).shape == (8, 4, 256)
print("Checkpoint Lab 4 passed")
```